# `02 — Basic Data Structures`

We study:
- Vector (dynamic array)
- Linked list
- Doubly linked list
- Stack (LIFO)
- Queue (FIFO)


Goal:
- Know what they look like in memory (contiguous vs pointers)
- Know operations + Big-O.
- Know which structure to choose for which task

---

## `1. Two "families"`

### A) Contiguous memory (Array / Vector)
> [  a0 | a1 | a2 | a3 | ... ]
- Fast random access by index (get/set): $O(1)$
- Insert/remove in the middle: shifts many elements → $O(n)$


### B) Linked memory (Linked lists)
> a0 -> a1 -> a2 -> a3 -> None
- Random access by index is slow: must walk nodes → $O(n)$
- Insert/remove near a known node can be $O(1)$ (depends on singly vs doubly)

---

## `2. Vector = Dynamic Array`


### `2.1 Intuition`:
We store items contiguously like an array, but we allow growth.

We keep:
- `len` = number of elements actually stored
- `capacity` = size of allocated memory

**Example** (len=5, capacity=11):
```bash
index:     0  1  2  3  4  5  6  7  8  9 10
storage:  [1][2][3][4][5][ ][ ][ ][ ][ ][ ]
                     len^        capacity^
```

`push_back(x)`:
- if len < capacity: put at a[len], len++
- else: allocate a new array of size 2*capacity, copy old elements, then insert


In [1]:
from dataclasses import dataclass
from typing import Self, Any, List, Optional


@dataclass
class VectorSim:

    capacity: int = 0
    length: int = 0
    data: Optional[List[Optional[Any]]] = None

    def _resize(self: Self, new_capacity: int) -> None:

        old = self.data if self.data is not None else []
        new: List[Optional[Any]] = [None] * new_capacity

        # Copy existing elements
        for i in range(self.length):
            new[i] = old[i]

        self.data = new
        self.capacity = new_capacity

    def push_back(self: Self, x: Any) -> None:

        if self.capacity == 0:
            self._resize(new_capacity=1)

        elif self.length == self.capacity:
            self._resize(new_capacity=self.capacity * 2)

        assert self.data is not None
        self.data[self.length] = x
        self.length += 1

    def pop_back(self: Self) -> Any:

        if self.length == 0:
            raise IndexError("pop_back from empty vector")
        
        assert self.data is not None
        x = self.data[self.length - 1]
        self.data[self.length - 1] = None
        self.length -= 1
        return x

    def __repr__(self: Self) -> str:

        assert self.data is not None
        return f"VectorSim(len={self.length}, cap={self.capacity}, data={self.data})"


v = VectorSim()

for x in range(10):
    v.push_back(x)
    print(v)

# Note: Capacity doubles when needed, like in the slides were mentioned.

VectorSim(len=1, cap=1, data=[0])
VectorSim(len=2, cap=2, data=[0, 1])
VectorSim(len=3, cap=4, data=[0, 1, 2, None])
VectorSim(len=4, cap=4, data=[0, 1, 2, 3])
VectorSim(len=5, cap=8, data=[0, 1, 2, 3, 4, None, None, None])
VectorSim(len=6, cap=8, data=[0, 1, 2, 3, 4, 5, None, None])
VectorSim(len=7, cap=8, data=[0, 1, 2, 3, 4, 5, 6, None])
VectorSim(len=8, cap=8, data=[0, 1, 2, 3, 4, 5, 6, 7])
VectorSim(len=9, cap=16, data=[0, 1, 2, 3, 4, 5, 6, 7, 8, None, None, None, None, None, None, None])
VectorSim(len=10, cap=16, data=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, None, None, None, None, None, None])


### `2.2 Vector operations and complexity`:

| Operation | Time Complexity |
|-----------|----------------|
| `get/set` by index | **O(1)** |
| `push_back` | **O(1) amortized**, **O(n) worst-case** (resize + copy) |
| `pop_back` | **O(1)** |
| `insert` in middle | **O(n)** (shift right) |
| `remove` in middle | **O(n)** (shift left) |
| `push_front` / `pop_front` | **O(n)** (same reason: shifting) |

---

## `3. Singly Linked List`

### `3.1 Intuition`:

Each node stores:
(value, next)

> head -> [val | next] -> [val | next] -> [val | None]

Key point:
- To reach the i-th element, you must walk i steps → get(i) is O(i) → worst O(n)
- Inserting *after a known node* is easy: just rewiring pointers → O(1)

### `3.2 Linked List operations and complexity`:

| Operation | Time Complexity |
|-----------|----------------|
| `push_front` | **O(1)** |
| `pop_front` | **O(1)** |
| `push_back` | **O(1)** (if we keep tail pointer) |
| `pop_back` | **O(n)** (need the previous of tail, must scan from head) |
| `get/set` by index | **O(n)** |
| `insert` after a node (if you already have the node) | **O(1)** |
| `remove` a node (in singly list, typically need previous) | **O(n)** if you only have node |

<br>

<div style="text-align: center;">
    <video width="1000" controls>
    <source src="./Videos/LinkedListSlide.mp4" type="video/mp4">
    </video>
</div>


In [15]:
from dataclasses import dataclass
from typing import Optional, Any, Self


@dataclass
class SNode:
    val: Any
    next: Optional["SNode"] = None


class SinglyLinkedList:

    def __init__(self: Self) -> None:
        self.head: Optional[SNode] = None
        self.tail: Optional[SNode] = None
        self.length: int = 0

    def push_front(self: Self, x: Any) -> None:
        node = SNode(val=x, next=self.head)
        self.head = node

        if self.tail is None:
            self.tail = node

        self.length += 1

    def push_back(self: Self, x: Any) -> None:
        node = SNode(val=x, next=None)

        if self.tail is None:
            self.head = self.tail = node
        else:
            self.tail.next = node
            self.tail = node

        self.length += 1

    def pop_front(self: Self) -> Any:
        if self.head is None:
            raise IndexError("pop_front from empty list")

        x = self.head.val
        self.head = self.head.next
        self.length -= 1

        if self.head is None:
            self.tail = None

        return x

    def pop_back(self: Self) -> Any:
        if self.head is None:
            raise IndexError("pop_back from empty list")

        if self.head.next is None:
            x = self.head.val
            self.head = self.tail = None
            self.length = 0
            return x

        # O(n): scan to find previous of tail
        prev = self.head
        while prev.next is not self.tail:
            assert prev.next is not None
            prev = prev.next

        assert self.tail is not None
        x = self.tail.val
        prev.next = None
        self.tail = prev
        self.length -= 1

        return x

    def to_ascii(self: Self) -> str:
        parts = []
        cur = self.head
        while cur is not None:
            parts.append(f"[{cur.val}]")
            cur = cur.next
        return " -> ".join(parts) + " -> None"

    def show(self: Self, label: str = "") -> None:
        head_val = None if self.head is None else self.head.val
        tail_val = None if self.tail is None else self.tail.val
        prefix = f"{label} " if label else ""
        print(f"{prefix}SinglyLinkedList(len={self.length}, head={head_val}, tail={tail_val})")
        print("   ", self.to_ascii())

 

In [26]:
# Example 01:

print('-' * 55)
print(f"{'Building the List':^55}")
print('-' * 55)

ll = SinglyLinkedList()
ll.show(label="Start")

for x in [1, 2, 3, 4, 5]:
    ll.push_back(x)
    ll.show(label=f"push_back({x})")

print('-' * 55)
print(f"{'Shrinking with pop_back()':^55}")
print('-' * 55)

while ll.length > 0:
    removed = ll.pop_back()
    ll.show(label=f"pop_back() -> {removed}")
   

-------------------------------------------------------
                   Building the List                   
-------------------------------------------------------
Start SinglyLinkedList(len=0, head=None, tail=None)
     -> None
push_back(1) SinglyLinkedList(len=1, head=1, tail=1)
    [1] -> None
push_back(2) SinglyLinkedList(len=2, head=1, tail=2)
    [1] -> [2] -> None
push_back(3) SinglyLinkedList(len=3, head=1, tail=3)
    [1] -> [2] -> [3] -> None
push_back(4) SinglyLinkedList(len=4, head=1, tail=4)
    [1] -> [2] -> [3] -> [4] -> None
push_back(5) SinglyLinkedList(len=5, head=1, tail=5)
    [1] -> [2] -> [3] -> [4] -> [5] -> None
-------------------------------------------------------
               Shrinking with pop_back()               
-------------------------------------------------------
pop_back() -> 5 SinglyLinkedList(len=4, head=1, tail=4)
    [1] -> [2] -> [3] -> [4] -> None
pop_back() -> 4 SinglyLinkedList(len=3, head=1, tail=3)
    [1] -> [2] -> [3] -> None
pop_b

In [25]:
# Example 02:

print('-' * 55)
print(f"{'Building the List':^55}")
print('-' * 55)

ll = SinglyLinkedList()
ll.show(label="Start")

for x in [1, 2, 3, 4, 5]:
    if x % 2 == 0:
        ll.push_front(x)
        ll.show(label=f"push_front({x})")
    else:
        ll.push_back(x)
        ll.show(label=f"push_back({x})")

print('-' * 55)
print(f"{'Shrinking with pop_back()':^55}")
print('-' * 55)

while ll.length > 0:
    removed = ll.pop_back()
    ll.show(label=f"pop_back() -> {removed}")
   

-------------------------------------------------------
                   Building the List                   
-------------------------------------------------------
Start SinglyLinkedList(len=0, head=None, tail=None)
     -> None
push_back(1) SinglyLinkedList(len=1, head=1, tail=1)
    [1] -> None
push_front(2) SinglyLinkedList(len=2, head=2, tail=1)
    [2] -> [1] -> None
push_back(3) SinglyLinkedList(len=3, head=2, tail=3)
    [2] -> [1] -> [3] -> None
push_front(4) SinglyLinkedList(len=4, head=4, tail=3)
    [4] -> [2] -> [1] -> [3] -> None
push_back(5) SinglyLinkedList(len=5, head=4, tail=5)
    [4] -> [2] -> [1] -> [3] -> [5] -> None
-------------------------------------------------------
               Shrinking with pop_back()               
-------------------------------------------------------
pop_back() -> 5 SinglyLinkedList(len=4, head=4, tail=3)
    [4] -> [2] -> [1] -> [3] -> None
pop_back() -> 3 SinglyLinkedList(len=3, head=4, tail=1)
    [4] -> [2] -> [1] -> None
pop

## `4. Doubly Linked List`


### `4.1 Intuition`:
Each node stores:
(prev, value, next)

> None <- [1] <-> [2] <-> [3] -> None

***Why it matters***:
* If you already have a pointer to a node X, you can remove it in O(1):
```bash
X.prev.next = X.next
X.next.prev = X.prev
```

This is exactly why doubly lists fix the "pop_back is O(n)" pain of singly lists.

### `4.2 Doubly Linked List operations and complexity`:

| Operation | Time Complexity |
|-----------|----------------|
| `push_front` | **O(1)** |
| `pop_front` | **O(1)** |
| `push_back` | **O(1)** |
| `pop_back` | **O(1)** |
| `get/set` by index | **O(n)** |
| `insert` after a given node (if you have the node) | **O(1)** |
| `remove` a given node (if you have the node) | **O(1)** |


In [27]:
from dataclasses import dataclass
from typing import Optional, Any, Self


@dataclass
class DNode:
    val: Any
    prev: Optional["DNode"] = None
    next: Optional["DNode"] = None


class DoublyLinkedList:

    def __init__(self: Self) -> None:
        self.head: Optional[DNode] = None
        self.tail: Optional[DNode] = None
        self.length: int = 0

    def push_back(self: Self, x: Any) -> DNode:
        node = DNode(val=x)

        if self.tail is None:
            self.head = self.tail = node
        else:
            node.prev = self.tail
            self.tail.next = node
            self.tail = node

        self.length += 1
        return node

    def push_front(self: Self, x: Any) -> DNode:
        node = DNode(val=x)

        if self.head is None:
            self.head = self.tail = node
        else:
            node.next = self.head
            self.head.prev = node
            self.head = node

        self.length += 1
        return node

    def pop_back(self: Self) -> Any:
        if self.tail is None:
            raise IndexError("pop_back from empty list")

        node = self.tail
        x = node.val

        if node.prev is None:
            # Only one element:
            self.head = self.tail = None
        else:
            node.prev.next = None
            self.tail = node.prev

        self.length -= 1
        return x

    def pop_front(self: Self) -> Any:
        if self.head is None:
            raise IndexError("pop_front from empty list")

        node = self.head
        x = node.val

        if node.next is None:
            # Only one element:
            self.head = self.tail = None
        else:
            node.next.prev = None
            self.head = node.next

        self.length -= 1
        return x

    def remove_node(self: Self, node: DNode) -> Any:

        # O(1) if node is given:
        if node.prev is not None:
            node.prev.next = node.next
        else:
            self.head = node.next

        if node.next is not None:
            node.next.prev = node.prev
        else:
            self.tail = node.prev

        self.length -= 1
        return node.val


    def to_ascii(self: Self) -> str:
        vals = []
        cur = self.head
        while cur is not None:
            vals.append(str(cur.val))
            cur = cur.next
        return "None <- " + " <-> ".join(vals) + " -> None"

    def show(self: Self, label: str = "") -> None:
        head_val = None if self.head is None else self.head.val
        tail_val = None if self.tail is None else self.tail.val
        prefix = f"{label} " if label else ""
        print(f"{prefix}DoublyLinkedList(len={self.length}, head={head_val}, tail={tail_val})")
        print("   ", self.to_ascii())



start DoublyLinkedList(len=0, head=None, tail=None)
    None <-  -> None
push_back(1) DoublyLinkedList(len=1, head=1, tail=1)
    None <- 1 -> None
push_back(2) DoublyLinkedList(len=2, head=1, tail=2)
    None <- 1 <-> 2 -> None
push_back(3) DoublyLinkedList(len=3, head=1, tail=3)
    None <- 1 <-> 2 <-> 3 -> None
push_back(4) DoublyLinkedList(len=4, head=1, tail=4)
    None <- 1 <-> 2 <-> 3 <-> 4 -> None
push_back(5) DoublyLinkedList(len=5, head=1, tail=5)
    None <- 1 <-> 2 <-> 3 <-> 4 <-> 5 -> None

--- pop_back (O(1)) ---
pop_back() -> 5 DoublyLinkedList(len=4, head=1, tail=4)
    None <- 1 <-> 2 <-> 3 <-> 4 -> None
pop_back() -> 4 DoublyLinkedList(len=3, head=1, tail=3)
    None <- 1 <-> 2 <-> 3 -> None

--- push_front (O(1)) ---
push_front(99) DoublyLinkedList(len=4, head=99, tail=3)
    None <- 99 <-> 1 <-> 2 <-> 3 -> None

--- pop_front (O(1)) ---
pop_front() -> 99 DoublyLinkedList(len=3, head=1, tail=3)
    None <- 1 <-> 2 <-> 3 -> None


In [35]:
# Example 01:
print('-' * 55)
print(f"{'Building the Doubly Linked List':^55}")
print('-' * 55)

dll = DoublyLinkedList()
dll.show(label="start")

for x in [1, 2, 3, 4, 5]:
    dll.push_back(x)
    dll.show(label=f"push_back({x})")

print('-' * 55)
print(f"{'pop_back (O(1))':^55}")
print('-' * 55)
for _ in range(2):
    removed = dll.pop_back()
    dll.show(label=f"pop_back() -> {removed}")

print('-' * 55)
print(f"{'push_front (O(1))':^55}")
print('-' * 55)
dll.push_front(99)
dll.show(label="push_front(99)")

print('-' * 55)
print(f"{'pop_front (O(1))':^55}")
print('-' * 55)
removed = dll.pop_front()
dll.show(label=f"pop_front() -> {removed}")


-------------------------------------------------------
            Building the Doubly Linked List            
-------------------------------------------------------
start DoublyLinkedList(len=0, head=None, tail=None)
    None <-  -> None
push_back(1) DoublyLinkedList(len=1, head=1, tail=1)
    None <- 1 -> None
push_back(2) DoublyLinkedList(len=2, head=1, tail=2)
    None <- 1 <-> 2 -> None
push_back(3) DoublyLinkedList(len=3, head=1, tail=3)
    None <- 1 <-> 2 <-> 3 -> None
push_back(4) DoublyLinkedList(len=4, head=1, tail=4)
    None <- 1 <-> 2 <-> 3 <-> 4 -> None
push_back(5) DoublyLinkedList(len=5, head=1, tail=5)
    None <- 1 <-> 2 <-> 3 <-> 4 <-> 5 -> None
-------------------------------------------------------
                    pop_back (O(1))                    
-------------------------------------------------------
pop_back() -> 5 DoublyLinkedList(len=4, head=1, tail=4)
    None <- 1 <-> 2 <-> 3 <-> 4 -> None
pop_back() -> 4 DoublyLinkedList(len=3, head=1, tail=3)
    

## `5. Stack (LIFO) and Queue (FIFO) interfaces`:

### `5.1 Stack (LIFO = Last In, First Out):`

**Definition**:

> `Stacks`are data structures that represents a dynamic set of data. We can `insert` and `delete` elements into a stack, but the delete operation is done in a ***predefined*** manner. For Stack this is called LIFO (Last in, First Out). And we call the `insert` operation push(), and `delete` operation pop().

Like coins in a glass or weight in a gym bar:
```bash
push(1), push(5), push(10)  -> top is 10
pop() returns 10, then 5, then 1
```

Operations:
- push(x) | $O(1)$ Complexity
- top()/front()
- pop() | $O(1)$ Complexity
- len()

If implemented with Vector/list (end as top):
- push: O(1) amortized
- pop: O(1)
- top: O(1)

**Use Cases**:
1. Backtracking:
    - Finding the correct path through a maze
2. Compile-Time memory management:
    - Programs use them to store local data and procedure info
    - Nested and recursive funcionts
3. Depth-First Search (DFP)


<div style="text-align: center;">
    <video width="1000" controls>
    <source src="./Videos/StackVisuals.mp4" type="video/mp4">
    </video>
</div>


In [38]:
from typing import List


# Stack using list (top is the end):
stack: List[int] = list()
stack.append(1)     # push
stack.append(5)
stack.append(10)

print('-' * 50)
print("stack:", stack)
print("pop:", stack.pop())  # 10
print("pop:", stack.pop())  # 5
print("stack after:", stack)

print('-' * 50)

--------------------------------------------------
stack: [1, 5, 10]
pop: 10
pop: 5
stack after: [1]
--------------------------------------------------


### `5.2 Queue (FIFO = First In, First Out):`

**Definition**:

> `Queue`are data structures that represents a dynamic set of data. We can `insert` and `delete` elements into a stack, but the delete operation is done in a ***predefined*** manner. For Queue this is called FIFO (First in, First Out). And we call the `insert` operation push()/enqueue(), and `delete` operation pop()/dequeue().

Like a shop queue:
```bash
push(1), push(3), push(5)
pop() returns 1, then 3, then 5
```

Operations:
- push(x)  (enqueue at back) $O(1)$ Complexity
- front()
- pop()    (dequeue from front) $O(1)$ Complexity
- len()

Important implementation note in Python:
- list.pop(0) is O(n) (shifts elements)
- collections.deque gives O(1) push/pop on both ends

**Use Cases**:
1. Operating Systems:
    - CPU and Disk Scheduling
2. Spotify or YouTube:
    - When we swap right in a song or video we add them into a queue

3. Breadth-First Search (BFP)

<div style="text-align: center;">
    <video width="1000" controls>
    <source src="./Videos/QueueVisuals.mp4" type="video/mp4">
    </video>
</div>

In [40]:
from typing import Deque
from collections import deque

# Queue using deque (front is left):
q: Deque[int] = deque()
q.append(1)         # push/enqueue
q.append(3)
q.append(5)

print('-' * 50)
print("queue:", q)
print("pop:", q.popleft())  # 1
print("pop:", q.popleft())  # 3
print("queue after:", q)
print('-' * 50)


--------------------------------------------------
queue: deque([1, 3, 5])
pop: 1
pop: 3
queue after: deque([5])
--------------------------------------------------


## **Quick comparison**:

| Operation | Vector | Singly Linked List | Doubly Linked List |
|---|---:|---:|---:|
| push_back | O(1) amortized | O(1) | O(1) |
| pop_back | O(1) | O(n) | O(1) |
| get/set by index | O(1) | O(n) | O(n) |
| insert in middle (by position) | O(n) | O(1)\* | O(1)\* |
| remove in middle (given node) | O(n) | O(n) | O(1) |
| push_front | O(n) | O(1) | O(1) |
| pop_front | O(n) | O(1) | O(1) |

\* Linked lists are O(1) **only if you already have the node** (or previous node).
Otherwise you pay O(n) to find it by index first.

## *References Extra Materials (videos)*:

1. [Linked Lists](https://www.youtube.com/watch?v=F8AbOfQwl1c&list=PL9xmBV_5YoZO2D89q42-y8voxIJKpB4oR)
2. [Stacks](https://www.youtube.com/watch?v=KcT3aVgrrpU&list=PL9xmBV_5YoZO2D89q42-y8voxIJKpB4oR&index=2)
3. [Queues](https://www.youtube.com/watch?v=D6gu-_tmEpQ&list=PL9xmBV_5YoZO2D89q42-y8voxIJKpB4oR&index=3)
